# 05 · Synthetischen Playground-Schreibtest ausführen

Eigenständig mit frischem Python-Kernel im Repository starten. **Run All führt
standardmäßig nur die Vorschau und lesende Prüfung aus.** Die letzte Zelle ist
deaktiviert. Keine echten GaSi-Messdaten werden eingelesen oder hochgeladen.

Die Laborbuch- und Inventarziele zentral mit
[GaSi_Playground_Ziele_einrichten.ipynb](GaSi_Playground_Ziele_einrichten.ipynb) einrichten.
Sie müssen ausdrücklich in der lokalen Datei
`config/playground_test_target.json` stehen und zum freigegebenen FB-8.1-Testbereich
gehören. Das Notebook rät keine Pfade und legt keine Spaces, Projekte oder Collections an.

In [ ]:
from gasi_openbis_playground import (
    load_target_config, show_plan, inspect_playground, check_write_target,
)

target_config, message = load_target_config()
print(message)
show_plan(target_config)

## Zentrale Session verwenden

Der Token kommt aus der lokalen `.env` beziehungsweise einer gesetzten
`OPENBIS_PLAYGROUND_TOKEN`-Umgebungsvariable. Zum Eintragen oder Erneuern
`GaSi_00_Session_einrichten.ipynb` verwenden. Ohne Token wird die Instanzprüfung
mit einem konkreten Hinweis übersprungen. Zugangswerte werden nicht ausgegeben.

In [ ]:
from gasi_session import session_status

print(session_status())

In [ ]:
report, message = inspect_playground(target_config)
print(message)
gate = check_write_target(report, target_config)
print("Ziel-/Typ-/Pflichtfeldprüfung:", "bereit für manuell aktivierten Test" if gate["ready"] else "gesperrt")
for reason in gate["reasons"]:
    print("- " + reason)
print("Schreibrechte und Server-Validatoren sind erst durch den Schreibtest bestätigt.")

## Bewusst aktivierter Schreibtest

Nach Prüfung der Vorschau `WRITE_ENABLED = True` und die Bestätigungsphrase setzen,
dann **nur diese Zelle** ausführen. Sie lädt die lokale Konfiguration neu und prüft
Host, beide Zielpfade, Typen, Pflichtfelder und Vokabularwerte erneut.

Genau ein synthetisches Instrument, ein Testmesslauf, Instrument als Parent und
eine synthetische CSV als `RAW_DATA`. Danach: Messlauf, Parent und Dataset neu lesen,
Datei herunterladen und SHA-256 vergleichen. IDs und Collection-Pfade erscheinen
im Ergebnis; im verlinkten Browser die jeweilige Collection öffnen bzw. nach ID suchen.

Wiederholungen verwenden dieselben Testkennungen. Nach einem Abbruch dieselbe
Zelle erneut ausführen, um fehlende Schritte zu ergänzen. Bei Konflikten oder
abweichendem Hash manuell prüfen. Immer nur einen Kernel gleichzeitig schreiben
lassen. Vor dem Speichern/Commit `WRITE_ENABLED` wieder auf `False` und die Phrase
auf `""` setzen sowie Notebook-Ausgaben löschen.

In [ ]:
WRITE_ENABLED = False
WRITE_CONFIRMATION = ""

if not WRITE_ENABLED:
    print("Schreibtest deaktiviert. Nur Vorschau und lesende Prüfung ausgeführt.")
elif WRITE_CONFIRMATION != "WRITE SYNTHETIC TEST TO BAM PLAYGROUND":
    print("Schreibtest gesperrt: Bestätigungsphrase fehlt.")
else:
    import json
    from gasi_openbis_playground import write_playground_test

    result, message = write_playground_test()
    print(message)
    if result is not None:
        print(json.dumps(result, ensure_ascii=False, indent=2))